# Pythia 1b

In [1]:
import sys
sys.path.insert(0, '../..')
from src.models import load_model, unload

### Load Model

In [2]:
model_name = "pythia-1b"
model, info = load_model(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-1b into HookedTransformer
Loaded pythia-1b on mps
  16 layers | 8 heads | d_model=2048 | d_mlp=8192 | parallel attn+MLP


### Baseline Prompt

In [3]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a program that reads text from a computer screen and

Cached 291 different activation points!


### Declarative Prompts

In [4]:
test_prompts = [
    "A screen reader is",
    "WCAG stands for", 
    "A skip link is",
    "The purpose of alt text is",
    "ARIA stands for",
    "A focus indicator is",
    "Keyboard navigation allows",
    "Color contrast is important because",
    "Semantic HTML helps",
    "Captions are used for",
]

for prompt in test_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a program that reads text from a computer screen and
WCAG stands for                →  what?

The World Council for Children’
A skip link is                 →  a type of data link used in a computer network
The purpose of alt text is     →  to provide a way to add text to an image
ARIA stands for                →  “Artificial Replacement of a Human Being”.
A focus indicator is           →  a device that is used to indicate the focus of
Keyboard navigation allows     →  you to navigate through the pages of this website.
Color contrast is important because →  it is the basis for many of the visual effects
Semantic HTML helps            →  you to create a website that is easy to read
Captions are used for          →  a variety of purposes, including providing information to a


### Evaluative Prompts

In [5]:
code_prompts = [
    "The following code is not accessible because it doesn't have what? <img src='photo.jpg'>",
    "A <div> with onclick is not accessible because",
    "The accessibility problem with <a href='#'></a> is",
    "<input type='text'> needs a",
    "A button that only says 'Click here' is bad because",
]
for prompt in code_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not a <span> element.

A <div> with onclick is not accessible
The accessibility problem with <a href='#'></a> is →  that it is not a valid HTML tag.

The <a href='#'></a>
<input type='text'> needs a    →  value
<input type='text'> needs a value
<input type='text'> needs a
A button that only says 'Click here' is bad because →  it's a clickable link.

A button that only says 'Click here' is bad


### Perplexity

In [6]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    
    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()
    
    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 18.804874420166016
Wrong: 42.179691314697266


### Attention Binding

In [7]:
import pandas as pd

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

out_path = (f"../results/{model_name}")
df = pd.DataFrame(rows)
df.to_csv(f"../results/{model_name}/attention-binding.csv", index=False)


print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")    

print(f"\nFound {len(rows)} heads above threshold")
print(f"Saved to {out_path}")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]

Top 10 by binding strength:
Layer  3, Head  5: 0.9932
Layer  1, Head  4: 0.9704
Layer  0, Head  3: 0.9656
Layer  1, Head  0: 0.8892
Layer  3, Head  6: 0.7256
Layer  2, Head  2: 0.7
Layer  1, Head  3: 0.6673
Layer  2, Head  0: 0.661
Layer  6, Head  3: 0.645
Layer  1, Head  2: 0.5844

Found 28 heads above threshold
Saved to ../results/pythia-1b


In [8]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 11
attention = cache["pattern", layer]  # shape: [heads, seq, seq]
print(f"Layer 3, Head 5")

html = cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

html

Layer 3, Head 5


In [9]:
unload(model)

Model unloaded, memory cleared
